# Age Estimation and Gender Classification

In this assignment, you will train CNN models to estimate a person's age and predict their gender based on face images. Please read carefully all the instructions before you start to write your code.

**Your Tasks**

You are supposed to train two CNN models.
- Model A: Train a CNN model from scratch, following a few restrictions. Save this model as `age_gender_A.keras`
- Model B: Fine-tune a pre-trained model to perform the same task. Save this model as `age_gender_B.keras`

**Dataset**

Your models will be trained and validated on a folder `train_val/` containing 5,000 labeled face images (size: 128 x 128), originated from the UTKFace dataset. During the assessment phase, your code will be tested on unseen test data.

**Performance Metric**

The evaluation of your models on the test set will be based on the following metrics:
- age estimation: - Mean Absolute Error (MAE)
- gender prediction: accuracy

**GPU Time Management**

Please use the available GPU time wisely, as there are some limitations on free GPU usage. It is said the users can use up to 12 hours in row in Google Colab. But many people found they reached its limit far less than 12 hours. To minimize the risk of reaching the limit, consider the following suggestions:
1. Ensure you have a stable internet connection.
2. Carefully validate your hyperparameters before starting the training process.
3. Based on past experience, each model should be trained in less than 2 hours. If the training time is significantly longer, you may need to adjust your model architecture.


## Setting Up: Mount the google drive
Mount your google drive to the notebook. Also don't forget to **enable GPU** before your training.


In [ ]:
#
# Add your code here
from google.colab import drive
drive.mount('/content/drive')
#

## Visualize a few photos
It's always beneficial to thoroughly understand your data before beginning the training process. To get a first-hand impression of the dataset, please display a minimum of 20 images from the `train_val/` folder, along with their corresponding age and gender labels. This will help you gain initial insights into the dataset.

In [ ]:
#
# Add your code here
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.image import imread
folder = '/content/drive/MyDrive/Colab Notebooks/train_val'
all_image_files = [file for file in os.listdir(folder) if file.lower().endswith(('.jpg'))]
random.shuffle(all_image_files)
few_image_files = all_image_files[:20]
random.shuffle(all_image_files)
plt.figure(figsize=(7,7))
i = 1
for filename in few_image_files:
    path = os.path.join(folder, filename)
    image = imread(path)
    plt.subplot(4, 5, i)
    plt.imshow(image)
    plt.axis("off")
    parts = filename.split("_")
    age = str(parts[0])
    if parts[1] == "0":
      gender = "M"
    else:
      gender = "F"
    plt.title(age + gender)
    i += 1
plt.show()
#

## Step 1: Data Pre-Processing
In this step, you will perform necessary pre-processing tasks before feeding the data into a CNN. The pre-processing includes:

1. Splitting the data into training and validation sets.
2. Loading image data, gender labels and age values, and rescaling pixel values (originally integers between 0 and 255) to the range [0, 1].
3. Data augmentation

The first two pre-processing steps have already been completed for you. Your task in this step is limited to:

3. Data augmentation

Please note:

- Do not modify any part of the provided code except the section marked with ########.
- Do not rescale the age values to the range [0,1], as the original values are used during the testing stage. Rescaling them would significantly impact testing performance.



In [ ]:
import os
import random
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, Sequential

folder_train_val = '/content/drive/MyDrive/Colab Notebooks/train_val'
all_image_files = [file for file in os.listdir(folder_train_val) if file.lower().endswith(('.jpg'))]

# Shuffle the dataset to ensure random distribution
random.seed(0)  # Ensure reproducibility
random.shuffle(all_image_files)

# Calculate the number of images for each set
n_train_val = len(all_image_files)
train_end = int(n_train_val * 0.8)

# Split the dataset
train_image_files = all_image_files[:train_end]
val_image_files = all_image_files[train_end:]
print(f"Number of images for training: {len(train_image_files)}")
print(f"Number of images for validation: {len(val_image_files)}")

def load_imgs_lables(dataset_path,filenames):
  print('load all image data, age and gender labels...')
  images = []
  age_labels = []
  gender_labels = []
  for current_file_name in filenames:
    img = cv2.imread(os.path.join(dataset_path, current_file_name))
    img = img / 255.0  # Normalize pixel values
    labels = current_file_name.split('_')
    age_label = int(labels[0])
    gender_label = int(labels[1])
    age_labels.append(age_label)
    gender_labels.append(gender_label)
    images.append(img)

  # Convert to NumPy arrays
  images = np.array(images)
  age_labels = np.array(age_labels)
  gender_labels = np.array(gender_labels)
  return images, age_labels, gender_labels

# load data from the training set
train_images, train_age, train_gender = load_imgs_lables(folder_train_val,train_image_files)

# load data from the validation set
val_images, val_age, val_gender = load_imgs_lables(folder_train_val,val_image_files)

#
######### Add your code for data augmentation here #########
data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomShear(0.1)
])
#

## STEP2A: Build your own CNN network
In this step, you will define your own CNN model for identifying gender and predicting age. Although there are two tasks, you only need to build one CNN model with two outputs. You may refer to online resources for solutions.

Here are the key requirements and restrictions for your network:
1.	The input size must be 128 x 128 x 3, which means you **should not resize** the original images. This is because my test code relies on this particular input size. Any other size will cause problem in the testing stage.
2.  Please treat the gender classification as a binary problem, i.e., **the output layer for the gender branch has only 1 unit**, instead of 2. Although it is technically correct to treat gender classification as a multi-class problem with 2 classes, i.e., 2 units in the last layer, the test code only supports a 1-unit last-layer configuration for the gender branch.
3.	The size of feature maps being fed to the first fully connected layer must be less than 10 x 10, while there is no number limitation about the depth.
4.	Incorporate techniques to prevent overfitting as appropriate.

At the end of your implementation, use `modelA.summary()` to display the model architecture and use `plot_model()` to visualize the architecture.

In [ ]:
#
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Flatten, BatchNormalization, Dropout, GlobalAveragePooling2D
from tensorflow.keras.regularizers import l1_l2, l2

inputs = Input(shape=(128, 128, 3))

modelA = Sequential()
modelA.add(inputs)
modelA.add(data_augmentation)
modelA.add(Conv2D(32, (3, 3), activation='relu'))
modelA.add(Conv2D(32, (3, 3), activation='relu'))
modelA.add(BatchNormalization(momentum=0.2))
modelA.add(MaxPooling2D((2, 2)))
modelA.add(Dropout(0.3))
modelA.add(Conv2D(32, (5, 5), activation='relu'))
modelA.add(BatchNormalization(momentum=0.2))
modelA.add(MaxPooling2D((2, 2)))
modelA.add(Dropout(0.3))
modelA.add(Conv2D(256, (3, 3), activation='relu'))
modelA.add(BatchNormalization(momentum=0.2))
modelA.add(MaxPooling2D((2, 2)))
modelA.add(Dropout(0.3))
modelA.add(Conv2D(256, (5, 5), activation='relu'))
modelA.add(BatchNormalization(momentum=0.2))
modelA.add(GlobalAveragePooling2D())

features = modelA(inputs)

gender = Dense(1024, kernel_regularizer=l2(0.01),activation = 'relu')(features)
gender = Dense(512, kernel_regularizer=l2(0.01),activation = 'relu')(gender)
gender = Dropout(0.5)(gender)
gender = Dense(1, activation = 'sigmoid',name='gender')(gender)

age = Dense(1024, kernel_regularizer=l1_l2(0.01,0.01), activation = 'relu')(features)
age = Dropout(0.5)(age)
age = Dense(512, kernel_regularizer=l1_l2(0.01,0.01), activation = 'relu')(age)
age = Dropout(0.5)(age)
age = Dense(1, activation = 'linear',name='age')(age)

modelA = Model(inputs=modelA.inputs,outputs=[gender,age])
#

modelA.summary(expand_nested=True)
from tensorflow.keras.utils import plot_model
plot_model(modelA, expand_nested=True, show_shapes=True,dpi=100)

## STEP3A: Compile and train your model
Compile and train your model here.
Save your model by `modelA.save(your_model_folder+"age_gender_A.keras")` after training.

**DON'T use any other name for your model file.** This is because my test code relies on this particular model name. Any other file name will cause problem in the testing stage.

**Save the model with `save()`** to save the entire model (including a model's architecture, weights, and training configuration) as a `.keras` zip archive. Don't save it with `save_weights()`.


In [ ]:
#
# Add your code here
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

seed = 123456

import random
random.seed(seed)

import numpy as np
np.random.seed(seed)

import tensorflow as tf
tf.random.set_seed(seed)

es_callback = EarlyStopping( monitor='val_loss',
  min_delta = 0.001,
  patience=20,
  restore_best_weights=True)

modelA.compile(loss={'gender':'binary_crossentropy',
                     'age':'mse'},
               loss_weights={'gender': 1, 'age': 0.4},
               optimizer = Adam(),
               metrics={'gender':['accuracy'],
                        'age':['mae']})

history = modelA.fit(x=train_images,
                     y = {'gender':train_gender,
                          'age':train_age},
                     validation_data = (val_images,{'gender': val_gender,
                                                    'age': val_age}),
                     epochs = 80,
                     callbacks=[es_callback],
                     batch_size = 32)
#
#modelA.save(folder_models+'age_gender_A.keras')

## STEP4A: Draw the learning curves
Draw four learning curve figures for model A as follows.
1.	The loss of the gender classification over the training and validation set
2.	The accuracy of the gender classification over the training and validation set
3.	The loss of the age estimation over the training and validation set
4.	The MAE of the age estimation over the training and validation set

In [ ]:
#
# Add your code here

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(20, 4))
fig.add_subplot(1, 4, 1)
plt.plot(history.history['gender_loss'], label='Train Gender Loss')
plt.plot(history.history['val_gender_loss'], label='Val Gender Loss')
plt.legend()
plt.grid(True)
plt.xlabel('epoch')

fig.add_subplot(1, 4, 2)
plt.plot(history.history['gender_accuracy'], label='Train Gender Accuracy')
plt.plot(history.history['val_gender_accuracy'], label='Val Gender Accuracy')
plt.legend()
plt.grid(True)
plt.xlabel('epoch')

fig.add_subplot(1, 4, 3)
plt.plot(history.history['age_loss'], label='Train Age Loss')
plt.plot(history.history['val_age_loss'], label='Val Age Loss')
plt.legend()
plt.grid(True)
plt.xlabel('epoch')

fig.add_subplot(1, 4, 4)
plt.plot(history.history['age_mae'], label='Train Age MAE')
plt.plot(history.history['val_age_mae'], label='Val Age MAE')
plt.legend()
plt.grid(True)
plt.xlabel('epoch')

plt.show()
#


## STEP2B: Build a CNN network based on a pre-trained model
Choose an existing CNN architecture pre-trained on ImageNet and fine-tune it on this dataset.

As with Model A, **do not resize the input image size**.
Ensure that the output layer for **the gender branch has only 1 unit**.

At the end of the cell, use `modelB.summary()` to display the model architecture and use `plot_model()` to visualize the architecture.


In [ ]:
#
from keras.applications import VGG16
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Flatten, BatchNormalization, Dropout, GlobalAveragePooling2D
from tensorflow.keras.regularizers import l1_l2, l2

inputs = Input(shape=(128, 128, 3))

conv_base = VGG16(weights='imagenet',
 include_top = False,
 input_shape=(128,128,3))

for layer in conv_base.layers[:6]:
    layer.trainable = False

modelB = Sequential()
modelB.add(inputs)
modelB.add(data_augmentation)
modelB.add(conv_base)
modelB.add(GlobalAveragePooling2D())

features = modelB(inputs,training=False)

gender = Dense(256, kernel_regularizer=l2(0.01),activation = 'relu')(features)
gender = Dropout(0.3)(gender)
gender = Dense(1, activation = 'sigmoid',name='gender')(gender)

age = Dense(256,kernel_regularizer=l1_l2(0.01,0.01), activation = 'relu')(features)
age = Dropout(0.3)(age)
age = Dense(128, kernel_regularizer=l1_l2(0.01,0.01), activation = 'relu')(age)
age = Dropout(0.3)(age)
age = Dense(1, activation = 'linear',name='age')(age)

modelB = Model(inputs=modelB.inputs,outputs=[gender,age])
# Add your code here
#

modelB.summary(expand_nested=True)
from tensorflow.keras.utils import plot_model
plot_model(modelB, expand_nested=True, show_shapes=True,dpi=100)

## STEP3B: Compile and train your model
Compile and train your model here.
Save your model to `age_gender_B.keras` after training.

**DON'T use any other name for your model file.** This is because my test code relies on this particular model name. Any other file name will cause problem in the testing stage.

**Also, save the model with `save()` instead of `save_weights()`.**


In [ ]:
#
# Add your code here
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping

seed = 123456789

import random
random.seed(seed)

import numpy as np
np.random.seed(seed)

import tensorflow as tf
tf.random.set_seed(seed)


es_callback = EarlyStopping( monitor='val_loss',
  min_delta = 0.001,
  patience=5,
  restore_best_weights=True)

modelB.compile(loss={'gender':'binary_crossentropy',
                     'age':'mse'},
               loss_weights={'gender': 1, 'age': 0.05},
               optimizer = SGD(learning_rate=1e-4,momentum=0.9),
               metrics={'gender':['accuracy'],
                        'age':['mae']})

history2 = modelB.fit(x=train_images,
                      y = {'gender':train_gender,
                           'age':train_age},
                      validation_data = (val_images,{'gender': val_gender,
                                                      'age': val_age}),
                      callbacks=[es_callback],
                      epochs = 50,
                      batch_size = 32)
#

# modelB.save(your_model_folder+"age_gender_B.keras")

## STEP4B: Draw the learning curves
Draw four learning curve figures for model B as follows.
1. The loss of the gender classification over the training and validation set
2. The accuracy of the gender classification over the training and validation set
3. The loss of the age estimation over the training and validation set
4. The MAE of the age estimation over the training and validation set

In [ ]:
#
# Add your code here
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(20, 4))
fig.add_subplot(1, 4, 1)
plt.plot(history2.history['gender_loss'], label='Train Gender Loss')
plt.plot(history2.history['val_gender_loss'], label='Val Gender Loss')
plt.legend()
plt.grid(True)
plt.xlabel('epoch')

fig.add_subplot(1, 4, 2)
plt.plot(history2.history['gender_accuracy'], label='Train Gender Accuracy')
plt.plot(history2.history['val_gender_accuracy'], label='Val Gender Accuracy')
plt.legend()
plt.grid(True)
plt.xlabel('epoch')

fig.add_subplot(1, 4, 3)
plt.plot(history2.history['age_loss'], label='Train Age Loss')
plt.plot(history2.history['val_age_loss'], label='Val Age Loss')
plt.legend()
plt.grid(True)
plt.xlabel('epoch')

fig.add_subplot(1, 4, 4)
plt.plot(history2.history['age_mae'], label='Train Age MAE')
plt.plot(history2.history['val_age_mae'], label='Val Age MAE')
plt.legend()
plt.grid(True)
plt.xlabel('epoch')

plt.show()
#


## STEP5: Evaluate the model on the test set
I will load your two models for testing. The test set will not be available before your submission.

The metrics for measuring the performance on the test set are:
- age estimation: MAE (Mean Absolute Error)
- gender classification: accuracy


In [ ]:
#
# Don't add code in this cell when submitting this file
models = "/content/drive/MyDrive/Colab Notebooks/models"
modelB.save(models+"age_gender_B.keras")

#